In [1]:
!pip install openai Ipython opencv-python ultralytics

  Using cached numpy-2.4.3-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.3-cp312-cp312-win_amd64.whl (12.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.17 requires numpy<2,>=1, but you have numpy 2.4.3 which is incompatible.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.4.3 which is incompatible.
langchain-openai 0.0.6 requires numpy<2,>=1, but you have numpy 2.4.3 which is incompatible.


In [2]:
from openai import OpenAI
from IPython.display import Image, display
import base64

In [3]:
client = OpenAI()

In [4]:
caption_system_prompt = '''
Your goal is to generate short, descriptive captions for images of items.
You will be provided with an item image and the name of that item and you will output a caption that captures the most important information about the item.
If there are multiple items depicted, refer to the name provided to understand which item you should describe.
Your generated caption should be short (1 sentence), and include only the most important information about the item.
The most important information could be: the type of item, the style (if mentioned), the material or color if especially relevant and/or any distinctive features.
Keep it short and to the point.
'''

async def get_caption(img_url, title):
    with open(img_url, "rb") as f:
        img_bytes = f.read()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=300,
    messages=[
        {
            "role": "system",
            "content": caption_system_prompt
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": title
                },
                # The content type should be "image_url" to use gpt-4-turbo's vision capabilities
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{img_b64}"
                    }
                },
            ],
        }
    ]
    )

    return response.choices[0].message.content

In [5]:
img_url="tiger.jpg"
img = Image(url=img_url)
display(img)

caption = await get_caption(img_url, "Jungle")
print(caption)

A close-up profile of a tiger, showcasing its distinctive orange and black striped fur, whiskers, and strong facial features.


In [6]:
img_url="chair.jpg"
img = Image(url=img_url)
display(img)
caption = await get_caption(img_url, "chair")
print(caption)

A modern bar stool with a sleek design featuring a brown wooden backrest and a chrome base, set in a stylish kitchen environment.


In [7]:
async def get_dense_caption(img_url, title):
    with open(img_url, "rb") as f:
        img_bytes = f.read()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # vision-capable model
        messages=[
            {"role": "system", "content": "You are a dense captioning agent. Generate a richly descriptive caption that includes entities, attributes, relations, and context."},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": title},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                ]
            }
        ]
    )
    return response.choices[0].message.content

In [8]:
img_url="denseimage.jpg"
img = Image(url=img_url)
display(img)
caption = await get_dense_caption(img_url, "densecaotion")
print(caption)

A bustling urban scene captures the chaotic yet vibrant atmosphere of Bangkok, Thailand. The view reveals a busy street overwhelmed with an array of vehicles, including numerous SUVs, sedans, and local tuk-tuks, all caught in a standstill traffic jam. The sunlight glimmers off the cars, creating a dynamic interplay of light and shadow. Above, large billboards display advertisements, including a prominent one for The Strokes, hinting at an upcoming event. A yellow bus and a classic red bus share the road with a mix of modern and older transport vehicles, showcasing the city's diverse transit options. Lively storefronts, including Sephora and other colorful shops, line the streets, while pedestrians navigate the sidewalks, further contributing to Bangkok's energetic ambiance.


In [9]:
import cv2
async def get_cv2_boundries(img_url, title):
    img = cv2.imread(img_url)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a dense captioning agent. Use object boundaries to enrich the description."},
            {"role": "user", "content": f"Objects detected: {title} at (x1,y1,x2,y2), foliage at (x3,y3,x4,y4)."}
        ]
    )
    return response.choices[0].message.content

In [10]:
img_url="denseimage.jpg"
caption = await get_cv2_boundries(img_url, "bus")
print(caption)

A bus occupies the scene, defined by its rectangular boundary from (x1, y1) to (x2, y2), showcasing its vibrant exterior against the backdrop. Surrounding it, lush foliage is visible, framed within the coordinates (x3, y3) to (x4, y4), adding a natural element that contrasts with the structured form of the bus. The interplay between the geometric shapes of the bus and the organic shapes of the leaves creates a dynamic visual scene.


In [11]:
import requests
url = "https://huggingface.co/ultralytics/YOLOv8/resolve/main/yolov8x.pt"
response = requests.get(url)
with open("yolov8x.pt", "wb") as f:
    f.write(response.content)

In [12]:
# Install YOLOv8
# pip install ultralytics
from ultralytics import YOLO
import cv2

# Load a pre-trained YOLOv8 model (trained on COCO dataset)
model = YOLO("yolov8x.pt")
# Run detection on an image
results = model("denseimage.jpg")
# Extract detections
vehicle_counts = {}
for r in results:
    for box in r.boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]
        if label in ["car", "bus", "truck", "motorcycle", "bicycle","person"]:
            vehicle_counts[label] = vehicle_counts.get(label, 0) + 1

print("Vehicle counts:", vehicle_counts)


image 1/1 D:\Ethans\Python\AI-Automation\denseimage.jpg: 448x640 4 persons, 22 cars, 1 motorcycle, 5 buss, 1 truck, 1566.4ms
Speed: 17.7ms preprocess, 1566.4ms inference, 21.4ms postprocess per image at shape (1, 3, 448, 640)
Vehicle counts: {'bus': 5, 'car': 22, 'person': 4, 'motorcycle': 1, 'truck': 1}


In [13]:
from ultralytics import YOLO
model = YOLO("yolov8x.pt")
# Run detection on an image
results = model("denseimage.jpg")
for r in results:
    r.show()   # display bounding boxes


image 1/1 D:\Ethans\Python\AI-Automation\denseimage.jpg: 448x640 4 persons, 22 cars, 1 motorcycle, 5 buss, 1 truck, 1275.0ms
Speed: 11.8ms preprocess, 1275.0ms inference, 1.8ms postprocess per image at shape (1, 3, 448, 640)


In [ ]:
for r in results:
    img_with_boxes = r.plot()   # returns numpy array with boxes drawn
    cv2.imshow("Detections", img_with_boxes)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Load YOLOv8 model (Nano or Extra-Large)
model = YOLO("yolov8n.pt")  # or "yolov8x.pt" training model install

# Run detection
results = model("denseimage.jpg")

# Get the first result (since results is a list)
annotated_img = results[0].plot()  # returns numpy array with boxes drawn

# Show inline with matplotlib
plt.figure(figsize=(12,8))
plt.imshow(annotated_img)
plt.axis("off")
plt.show()



detections = []
for box in results[0].boxes:
    cls_id = int(box.cls[0])
    label = model.names[cls_id]
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    detections.append(f"{label} at ({int(x1)},{int(y1)},{int(x2)},{int(y2)})")

detections_str = "Objects detected: " + ", ".join(detections)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a dense captioning agent. Generate a richly descriptive caption based on detected objects and their positions."},
        {"role": "user", "content": detections_str}
    ]
)

print("Dense Caption:", response.choices[0].message.content)
